# Create Issue Indices for Impresso-1 BNF and BNF-EN Titles

Generates issue-index JSON shards for the **11 legacy titles** already imported
in Impresso-1:

| Set | Batch tag | Titles | Has OLR |
|-----|-----------|--------|---------|
| 1 – Marché Presse | `BNF` | Excelsior, La-Fronde, Marie-Claire, Oeuvre | yes |
| 0 – Europeana | `BNF-EN` | legaulois, lematin, lepji, jdpl, oecaen, oerennes, lepetitparisien | yes |

Index schema (same as `issue_index.bnf.json`):
```json
{
  "day": "15", "edition": "a",
  "local_path": ["BNF/Excelsior/4600000"],
  "ark_id": "bpt6k46000007",
  "batch": "BNF"
}
```

**Ark strategy per set**
- *BNF*: extracted directly from `<fileIdentifier>` in the first ALTO file in `ocr/` — no API call.
- *BNF-EN*: fetched from the Gallica API via the existing `get_issues_iiif_arks` helper;
  the METS file contains no ark information.

## Imports and configuration

In [4]:
import gzip
import json
import logging
import os
import re
import string
import sys
import requests
from dotenv import load_dotenv
from collections import defaultdict
from datetime import datetime, date
from pathlib import Path

from bs4 import BeautifulSoup
from tqdm import tqdm
load_dotenv()

REPO_ROOT = Path(__file__).resolve().parents[3] if "__file__" in dir() else Path.cwd().parents[2]
sys.path.insert(0, str(REPO_ROOT))

# BNF helpers used only for date parsing (manifest.xml)
from text_preparation.importers.bnf.helpers import get_journal_name, parse_date
from text_preparation.importers.bnf.detect import DATE_FORMATS, DATE_SEPARATORS
from text_preparation.importers.mets_alto.mets import get_dmd_sec

# BNF-EN: API helper to fetch issue-level ark IDs per journal
from text_preparation.importers.bnf_en.detect import API_MAPPING, get_issues_iiif_arks

logging.basicConfig(level=logging.WARNING)

ORIGINAL_BASE = "/mnt/project_impresso/original"
INDEX_OUT_DIR = REPO_ROOT / "text_preparation/data/issue_indices"
INDEX_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root  :", REPO_ROOT)
print("Output dir :", INDEX_OUT_DIR)

Repo root  : /home/piconti/impresso-text-acquisition
Output dir : /home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices


## Part 1 – BNF (Marché Presse) — 4 legacy titles

On-disk layout:
```
BNF/<JournalDir>/<numeric_id>/
    manifest.xml   ← date + secondary_date live here
    ocr/
        X0000001.xml.gz   ← fileIdentifier holds the issue-level ark
    toc/
```

The issue-level ark is at:
`<fileIdentifier>ark:/12148/bpt6k46000007/f1</fileIdentifier>`
→ `ark_id = bpt6k46000007`

Same-day multi-edition issues are sorted by numeric folder id and assigned
edition letters `a`, `b`, `c`… in that order.

In [ ]:
BNF_BASE_DIR = os.path.join(ORIGINAL_BASE, "BNF")
BNF_OUT_FILE = INDEX_OUT_DIR / "issue_index.bnf_mp.json"

LEGACY_BNF_DIRS = {
    "Excelsior":    "excelsior",
    "La-Fronde":    "lafronde",
    "Marie-Claire": "marieclaire",
    "Oeuvre":       "oeuvre",
}

In [3]:
def parse_bnf_manifest(manifest_path: str) -> tuple[date, date | None] | None:
    """Extract (date, secondary_date) from a BNF manifest.xml.

    Returns None when the file is missing or unparseable.
    The date is in dmdSec[@ID='DMD.2'] → mods:originInfo → mods:dateIssued.
    Delegates to the existing parse_date helper which handles the two BNF date
    formats (YYYY-MM-DD and YYYY/MM/DD) and dual-date issues.
    """
    try:
        with open(manifest_path, encoding="utf-8") as f:
            soup = BeautifulSoup(f, "xml")
        dmd2 = get_dmd_sec(soup, "DMD.2")
        if dmd2 is None:
            logging.warning("No DMD.2 in %s", manifest_path)
            return None
        raw_date = dmd2.find("date").contents[0]
        return parse_date(raw_date, DATE_FORMATS, DATE_SEPARATORS)
    except Exception as exc:
        logging.warning("Could not parse manifest %s: %s", manifest_path, exc)
        return None


def extract_issue_ark(ocr_dir: str) -> str | None:
    """Return the bare issue-level ark from the first ALTO file in ocr_dir.

    Format in file: <fileIdentifier>ark:/12148/bpt6k46000007/f1</fileIdentifier>
    Returns: 'bpt6k46000007'
    """
    try:
        alto_files = sorted(f for f in os.listdir(ocr_dir) if ".xml" in f)
        if not alto_files:
            logging.warning("No ALTO files in %s", ocr_dir)
            return None
        alto_path = os.path.join(ocr_dir, alto_files[0])
        opener = gzip.open if alto_path.endswith(".gz") else open
        with opener(alto_path) as fh:
            soup = BeautifulSoup(fh.read(), "xml")
        tag = soup.find("fileIdentifier")
        if tag is None:
            return None
        m = re.search(r"ark:/12148/([^/\s]+)", tag.get_text(strip=True))
        return m.group(1) if m else None
    except Exception as exc:
        logging.warning("ark extraction failed for %s: %s", ocr_dir, exc)
        return None


# Smoke-tests
_ark = extract_issue_ark("/mnt/project_impresso/original/BNF/Excelsior/4600000/ocr")
print("ark smoke-test :", _ark)          # → bpt6k46000007

_dt = parse_bnf_manifest("/mnt/project_impresso/original/BNF/Excelsior/4600000/manifest.xml")
print("date smoke-test:", _dt)           # → (date(1910, 11, 16), None)

ark smoke-test : bpt6k46000007
date smoke-test: (datetime.date(1910, 11, 16), None)


In [ ]:
def build_bnf_index(bnf_base: str, legacy_dirs: dict[str, str]) -> dict:
    """Build alias→year→month→[entries] for the 4 BNF Marché Presse legacy titles."""

    # Step 1 — collect raw records
    raw: list[dict] = []
    for dirname, alias in legacy_dirs.items():
        journal_path = os.path.join(bnf_base, dirname)
        if not os.path.isdir(journal_path):
            print(f"WARNING: {journal_path} not found — skipping")
            continue
        numeric_ids = sorted(d for d in os.listdir(journal_path) if d.isdigit())
        for num_id in tqdm(numeric_ids, desc=alias, leave=False):
            issue_path = os.path.join(journal_path, num_id)
            manifest   = os.path.join(issue_path, "manifest.xml")
            result     = parse_bnf_manifest(manifest)
            if result is None:
                continue
            np_date, secondary_date = result
            raw.append({
                "alias":          alias,
                "date":           np_date,
                "secondary_date": secondary_date,
                "path":           issue_path,
                "num_id":         num_id,
            })

    # Step 2 — assign edition letters per (alias, date), ordered by numeric id
    by_day: dict[tuple, list[dict]] = defaultdict(list)
    for r in raw:
        by_day[(r["alias"], r["date"])].append(r)
    for group in by_day.values():
        group.sort(key=lambda x: x["num_id"])
        for idx, r in enumerate(group):
            r["edition"] = string.ascii_lowercase[idx]

    # Step 3 — build index entries (extract ark from ALTO while we are here)
    index: dict = {}
    for r in tqdm(raw, desc="Building BNF entries"):
        alias = r["alias"]
        year  = str(r["date"].year)
        month = f"{r['date'].month:02d}"

        ocr_dir = os.path.join(r["path"], "ocr")
        ark_id  = extract_issue_ark(ocr_dir) if os.path.isdir(ocr_dir) else None
        rel     = os.path.relpath(r["path"], ORIGINAL_BASE)

        entry: dict = {
            "day":        f"{r['date'].day:02d}",
            "edition":    r["edition"],
            "local_path": [rel],
            "ark_id":     ark_id,
            "batch":      "BNF",
        }
        if r["secondary_date"] is not None:
            entry["secondary_date"] = str(r["secondary_date"])

        index.setdefault(alias, {}).setdefault(year, {}).setdefault(month, []).append(entry)

    # Step 4 — sort within each month
    for alias in index:
        for year in index[alias]:
            for month in index[alias][year]:
                index[alias][year][month].sort(key=lambda e: (e["day"], e["edition"]))

    return index


bnf_index = build_bnf_index(BNF_BASE_DIR, LEGACY_BNF_DIRS)

for alias, years in bnf_index.items():
    total = sum(len(es) for m in years.values() for es in m.values())
    print(f"  {alias}: {len(years)} years, {total} issues")

In [5]:
with open(BNF_OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(bnf_index, f, indent=2, ensure_ascii=False)
print(f"Saved → {BNF_OUT_FILE}")

alias = next(iter(bnf_index))
year  = next(iter(bnf_index[alias]))
month = next(iter(bnf_index[alias][year]))
print("\nSample entry:")
print(json.dumps(bnf_index[alias][year][month][0], indent=2))

Saved → /home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issue_index.bnf_1.json

Sample entry:
{
  "day": "16",
  "edition": "a",
  "local_path": [
    "BNF/Excelsior/4600000"
  ],
  "ark_id": "bpt6k46000007",
  "batch": "BNF"
}


## Part 2 – BNF-EN (Europeana) — 7 legacy titles

On-disk layout:
```
BNF-EN/old/<JournalDir>/<YYYYMMDD>_<edition_int>/
    <YYYYMMDD>_<ed>-METS.xml
    ALTO/
        <YYYYMMDD>_<ed>-0001.xml  …
```

Date and edition come directly from the directory name.
The METS file contains **no ark information**.
Issue-level arks are fetched once from the Gallica API using `get_issues_iiif_arks`;
the function returns `[(canonical_issue_id, ark_id)]` pairs for a given journal.

#### Fetch the access token

In [9]:
# url to get the token
TOKEN_URL = "https://apimauthproext.bnf.fr/oauth2/token"

# to be used as follows: 
# curl -X POST "https://apimauthproext.bnf.fr/oauth2/token" -u "KEY:SECRET" -d "grant_type=client_credentials"

def get_access_token(token_url=TOKEN_URL):
    key = os.environ["BNF_API_KEY"]
    secret = os.environ["BNF_API_SECRET"]
    response = requests.post(
        token_url,
        auth=(key, secret),
        data={"grant_type": "client_credentials"},
    )
    response.raise_for_status()
    return response.json()


In [10]:
response = get_access_token()
ACCESS_TOKEN = response["access_token"]

Try some things out

In [23]:
API_JOURNAL_URL = "https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/{ark}/date"

def get_issue_arks_for_title(title_ark_id, token=ACCESS_TOKEN, title_url=API_JOURNAL_URL):
    url = title_url.format(ark=title_ark_id)
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}"},
    )
    response.raise_for_status()
    return response

In [ ]:
url = "https://gallica.bnf.fr/services/Issues?ark=cb32830550k/date&date=1906"
r = requests.get(url, headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},)
r 

#### Fetch the data

In [2]:
BNFEN_BASE_DIR = os.path.join(ORIGINAL_BASE, "BNF-EN", "old")
BNFEN_OUT_FILE = INDEX_OUT_DIR / "issue_index.bnf_en1.json"

EDITION_NUM_TO_LETTER = {i: string.ascii_lowercase[i - 1] for i in range(1, 10)}

def dirname_to_alias(dirname: str) -> str:
    """Mirror the normalisation in bnf_en/detect.py dir2issue."""
    return dirname.lower().replace("-", "").strip()

# Sanity-check
EXPECTED = set(API_MAPPING.keys())
found = {
    dirname_to_alias(d)
    for d in os.listdir(BNFEN_BASE_DIR)
    if os.path.isdir(os.path.join(BNFEN_BASE_DIR, d)) and not d.startswith(".")
}
print("Found  :", found)
print("Missing:", EXPECTED - found)
print("Extra  :", found - EXPECTED)

Found  : {'legaulois', 'jdpl', 'oecaen', 'lematin', 'lepetitparisien', 'oerennes', 'lepji'}
Missing: set()
Extra  : set()


In [5]:
ark_map: dict[str, str] = {}

#def fetch_bnfen_arks(ark_map) -> dict[str, str]:
"""Fetch issue-level ark IDs for all 7 BNF-EN titles from the Gallica API.

Uses get_issues_iiif_arks (bnf_en/detect.py), which calls the Gallica
/services/Issues endpoint for each year.  Expect ~1-2 minutes total.

Returns:
    dict mapping canonical_issue_id → bare_ark_id
    e.g. {'legaulois-1884-04-08-a': 'bpt6k532073c', ...}
"""
for alias, journal_ark in tqdm(API_MAPPING.items(), desc="Fetching arks via Gallica API"):
    pairs = get_issues_iiif_arks((alias, journal_ark))
    for canonical_id, ark in pairs:
        ark_map[canonical_id] = ark
print(f"Fetched {len(ark_map)} issue arks across {len(API_MAPPING)} titles")
    #return ark_map

#bnfen_arks = fetch_bnfen_arks(ark_map)

Fetching arks via Gallica API:   0%|          | 0/7 [00:00<?, ?it/s]


Fetching for oerennes


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

A priori we're having problems to fetch this specific info using the previous API. 

I Have not found the new versions of these requests yet, so another option might be to fetch the ARK id and relevant information from the already generated canonical data which currently lives on S3.

In [ ]:
def parse_bnfen_dirname(dirname: str) -> tuple[date, str] | None:
    """Parse '{YYYYMMDD}_{edition_int}' → (date, edition_letter) or None."""
    m = re.fullmatch(r"(\d{8})_(\d+)", dirname)
    if not m:
        return None
    try:
        d = datetime.strptime(m.group(1), "%Y%m%d").date()
        edition = EDITION_NUM_TO_LETTER.get(int(m.group(2)),
                                            string.ascii_lowercase[int(m.group(2)) - 1])
        return d, edition
    except (ValueError, IndexError) as exc:
        logging.warning("Cannot parse dirname %s: %s", dirname, exc)
        return None


def build_bnfen_index(base_dir: str, arks: dict[str, str]) -> dict:
    """Build alias→year→month→[entries] for the 7 BNF-EN Europeana legacy titles."""
    index: dict = {}
    missing_arks: list[str] = []

    for journal_dirname in sorted(
        d for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d)) and not d.startswith(".")
    ):
        alias        = dirname_to_alias(journal_dirname)
        journal_path = os.path.join(base_dir, journal_dirname)

        for issue_dirname in tqdm(sorted(os.listdir(journal_path)), desc=alias, leave=False):
            issue_path = os.path.join(journal_path, issue_dirname)
            if not os.path.isdir(issue_path):
                continue
            parsed = parse_bnfen_dirname(issue_dirname)
            if parsed is None:
                logging.warning("Skipping unrecognised dir: %s", issue_path)
                continue
            issue_date, edition = parsed

            canonical_id = f"{alias}-{issue_date.year}-{issue_date.month:02d}-{issue_date.day:02d}-{edition}"
            ark_id = arks.get(canonical_id)
            if ark_id is None:
                missing_arks.append(canonical_id)

            entry = {
                "day":        f"{issue_date.day:02d}",
                "edition":    edition,
                "local_path": [os.path.relpath(issue_path, ORIGINAL_BASE)],
                "ark_id":     ark_id,
                "batch":      "BNF-EN",
            }
            year  = str(issue_date.year)
            month = f"{issue_date.month:02d}"
            index.setdefault(alias, {}).setdefault(year, {}).setdefault(month, []).append(entry)

    for alias in index:
        for year in index[alias]:
            for month in index[alias][year]:
                index[alias][year][month].sort(key=lambda e: (e["day"], e["edition"]))

    if missing_arks:
        print(f"WARNING: {len(missing_arks)} issues had no API ark — ark_id will be null")
        for cid in missing_arks[:10]:
            print(" ", cid)
        if len(missing_arks) > 10:
            print(f"  ... and {len(missing_arks) - 10} more")

    return index


bnfen_index = build_bnfen_index(BNFEN_BASE_DIR, bnfen_arks)

for alias, years in bnfen_index.items():
    total = sum(len(es) for m in years.values() for es in m.values())
    print(f"  {alias}: {len(years)} years, {total} issues")

In [ ]:
with open(BNFEN_OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(bnfen_index, f, indent=2, ensure_ascii=False)
print(f"Saved → {BNFEN_OUT_FILE}")

alias = next(iter(bnfen_index))
year  = next(iter(bnfen_index[alias]))
month = next(iter(bnfen_index[alias][year]))
print("\nSample entry:")
print(json.dumps(bnfen_index[alias][year][month][0], indent=2))

## Validation

In [ ]:
def count_index(index: dict) -> dict:
    issues, missing_arks = 0, 0
    years: set = set()
    for alias, yd in index.items():
        years |= set(yd)
        for year, md_ in yd.items():
            for month, entries in md_.items():
                issues += len(entries)
                missing_arks += sum(1 for e in entries if e.get("ark_id") is None)
    return {"aliases": len(index), "issues": issues,
            "years": sorted(years), "missing_arks": missing_arks}


def check_duplicates(index: dict, label: str) -> None:
    seen: set = set()
    dupes: list = []
    for alias, yd in index.items():
        for year, md_ in yd.items():
            for month, entries in md_.items():
                for e in entries:
                    key = (alias, year, month, e["day"], e["edition"])
                    if key in seen:
                        dupes.append(key)
                    seen.add(key)
    if dupes:
        print(f"[{label}] WARNING: {len(dupes)} duplicate keys!")
        for d in dupes[:5]:
            print(" ", d)
    else:
        print(f"[{label}] No duplicates — OK")


for label, idx in [("BNF", bnf_index), ("BNF-EN", bnfen_index)]:
    c = count_index(idx)
    print(f"=== {label} ===")
    print(f"  Aliases      : {c['aliases']}")
    print(f"  Issues       : {c['issues']}")
    print(f"  Year range   : {c['years'][0]} – {c['years'][-1]}")
    print(f"  Missing arks : {c['missing_arks']}")
    check_duplicates(idx, label)
    print()
print("Output files:")
print(f"  {BNF_OUT_FILE}")
print(f"  {BNFEN_OUT_FILE}")

### Mapping each alias to its BNF data format